# Predictive Classification using Random Forest Ensembles
**Student:** KOYYANA PARISRITH REDDY  
**Dataset:** Titanic passenger survival (Kaggle competition dataset)  
**Target:** `Survived` (0 = did not survive, 1 = survived)

## Project overview
This notebook follows the required workflow: exploratory data analysis, missing-value handling, categorical encoding, an 80:20 train/test split, Random Forest training, and evaluation with a confusion matrix, precision, recall, F1-score, ROC-AUC, and feature importance.

**Data source:** A public mirror of the Kaggle Titanic training data. If the URL is unavailable, upload `train.csv` from the Kaggle Titanic competition when prompted.

In [ ]:
# Import libraries
import io
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, classification_report
)
from IPython.display import display

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Load the dataset
The code first attempts to read a public mirror. If the request fails, it asks you to upload the Kaggle `train.csv` file.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Dataset loaded from public mirror.")
except Exception as exc:
    print("Could not download the dataset:", exc)
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
        if csv_name is None:
            raise ValueError("Please upload a CSV file, preferably Kaggle Titanic train.csv.")
        df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
    except ImportError:
        csv_path = input("Enter the path to your Titanic CSV file: ").strip()
        df = pd.read_csv(csv_path)

# Normalize common Kaggle column names if needed
if "Survived" not in df.columns and "survived" in df.columns:
    df = df.rename(columns={"survived": "Survived"})
if "Survived" not in df.columns:
    raise ValueError("Target column 'Survived' was not found. Use the Kaggle Titanic training dataset.")

print("Dataset shape:", df.shape)
display(df.head())

## 2. Exploratory Data Analysis (EDA)
Inspect data types, summary statistics, missing values, target balance, feature distributions, and a numeric correlation heatmap.

In [ ]:
display(df.info())
display(df.describe(include="all").T)
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_values"))

plt.figure(figsize=(6, 4), dpi=140)
sns.countplot(data=df, x="Survived")
plt.title("Target Class Distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.tight_layout()
plt.show()

numeric_cols = df.select_dtypes(include=np.number).columns
if len(numeric_cols) > 1:
    plt.figure(figsize=(10, 7), dpi=140)
    sns.heatmap(df[numeric_cols].corr(), cmap="coolwarm", center=0, annot=False)
    plt.title("Correlation Heatmap (Numeric Features)")
    plt.tight_layout()
    plt.show()

# Show a few useful feature distributions when present
for col in ["Age", "Fare", "Pclass"]:
    if col in df.columns:
        plt.figure(figsize=(6, 4), dpi=140)
        sns.histplot(data=df, x=col, hue="Survived", kde=(col != "Pclass"), bins=25)
        plt.title(f"{col} Distribution by Survival")
        plt.tight_layout()
        plt.show()

## 3. Preprocessing and train/test split
- Remove identifier/text fields that would not be useful as direct predictors (`PassengerId`, `Name`, `Ticket`, `Cabin`).
- Keep remaining numerical and categorical features.
- Impute numerical missing values with the median and categorical missing values with the most frequent category.
- One-hot encode categorical features.
- Split into 80% training and 20% testing data, stratifying by the target.
- Feature scaling is skipped because Random Forest uses decision-tree splits and is not distance-based.

In [ ]:
df_model = df.copy().dropna(subset=["Survived"])
y = df_model["Survived"].astype(int)

drop_cols = [c for c in ["Survived", "PassengerId", "Name", "Ticket", "Cabin"] if c in df_model.columns]
X = df_model.drop(columns=drop_cols)

# Drop any columns that contain no usable values
X = X.dropna(axis=1, how="all")

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 4. Train the Random Forest model
`n_estimators=300` provides a reasonably stable ensemble; `max_depth=8` limits tree complexity to reduce overfitting. `class_weight='balanced'` helps account for unequal class counts.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Model training complete.")

## 5. Required evaluation metrics
Interpret precision as how often positive predictions are correct, and recall as how many actual survivors are identified. F1 balances precision and recall. ROC-AUC measures ranking ability across thresholds.

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
metrics_df = pd.DataFrame({"Metric": metrics.keys(), "Score": metrics.values()})
display(metrics_df.style.format({"Score": "{:.4f}"}))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5), dpi=180)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted No", "Predicted Yes"],
            yticklabels=["Actual No", "Actual Yes"])
plt.title("Confusion Matrix")
plt.ylabel("Actual class")
plt.xlabel("Predicted class")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_value = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6, 5), dpi=180)
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance after preprocessing
fitted_preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()
importance = pd.Series(classifier.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6), dpi=180)
sns.barplot(x=importance.values, y=importance.index)
plt.title("Top 15 Feature Importances")
plt.xlabel("Mean decrease in impurity (importance)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(importance.rename("importance").to_frame())

## 6. Automatic interpretation and conclusion
The statements below are generated from the test-set scores so the written interpretation matches the actual run. Feature importance indicates predictive contribution in this fitted model; it does not prove causation.

In [ ]:
print("INTERPRETATION")
print(f"Accuracy: {metrics['Accuracy']:.3f} — fraction of test examples classified correctly.")
print(f"Precision: {metrics['Precision']:.3f} — of passengers predicted to survive, this fraction actually survived.")
print(f"Recall: {metrics['Recall']:.3f} — of passengers who survived, this fraction was identified.")
print(f"F1-score: {metrics['F1-score']:.3f} — balances precision and recall.")
print(f"ROC-AUC: {metrics['ROC-AUC']:.3f} — higher values indicate better separation of the two classes.")

if metrics["Recall"] < metrics["Precision"]:
    print("Recall is lower than precision: the model misses a comparatively larger share of actual survivors.")
elif metrics["Precision"] < metrics["Recall"]:
    print("Precision is lower than recall: a comparatively larger share of predicted survivors are false positives.")
else:
    print("Precision and recall are equal or very close on this test split.")

print("\nStrengths: handles mixed feature types after preprocessing, models nonlinear patterns, and provides feature importance.")
print("Limitations: performance depends on the split and available features; feature importance can be biased and is not causal.")
print("Improvements: cross-validated hyperparameter tuning, threshold adjustment, comparison with Gradient Boosting, and careful feature engineering.")

## 7. Save evaluation outputs
This cell saves the metrics table and feature-importance table as CSV files for use in the technical report.

In [ ]:
metrics_df.to_csv("random_forest_metrics.csv", index=False)
importance.rename("importance").to_csv("random_forest_feature_importance.csv", header=True)
print("Saved: random_forest_metrics.csv and random_forest_feature_importance.csv")